<a href="https://colab.research.google.com/github/amigli/Q-Bert_RL/blob/main/Notebook/PPO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Training e Test con PPO
In questo notebook è presente il training di Q*Bert sfruttando l'algoritmo PPO

## Download Repository

In [1]:
from google.colab import userdata

In [2]:
!git clone https://{userdata.get('TokenGithub')}"@github.com/amigli/Q-Bert_RL.git"

Cloning into 'Q-Bert_RL'...
remote: Enumerating objects: 598, done.
remote: Counting objects: 100% (70/70), done.
remote: Compressing objects: 100% (53/53), done.
remote: Total 598 (delta 40), reused 27 (delta 17), pack-reused 528 (from 1)
Receiving objects: 100% (598/598), 10.46 MiB | 10.21 MiB/s, done.
Resolving deltas: 100% (377/377), done.


In [3]:
%cd Q-Bert_RL/

/content/Q-Bert_RL


## Installazione dei requirements

In [4]:
!pip install gymnasium

In [5]:
!pip install ale-py

In [6]:
!pip install moviepy

In [7]:
!pip install stable-baselines3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 64.4 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalli

In [8]:
!pip install wandb

## Configurazione del salvataggio dei video

In [9]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [10]:
directory_videos = '/content/Videos/'

## Algoritmo

In [22]:
import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.evaluation import evaluate_policy
import ale_py
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import tensorflow as tf
from stable_baselines3.common.logger import configure
import torch
from torch.utils.tensorboard import SummaryWriter
from stable_baselines3.common.vec_env import VecVideoRecorder, DummyVecEnv
from EnvironmentWrappers.RewardFunction import RewardFunction
from EnvironmentWrappers.ObsRewardWrapper import ObsRewardWrapper
import wandb
from stable_baselines3.common.callbacks import BaseCallback, EvalCallback
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.utils import get_linear_fn
import torch as th



/usr/local/lib/python3.11/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


## Wandb

In [12]:
!wandb login

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit: 
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: frank581-fgz (frankzamma) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [72]:
# Inizializzazione di wandb
wandb.init(
    project="QBERT-RL",
    entity = "Q-BertRLTeam",

    config={
        "learning_rate": 0.0003,
        "epochs": 10,
    }
)

## Training

In [16]:
gym.register_envs(ale_py)

def make_env(env_id):
    def _init():
        env = gym.make(env_id)
        env = ObsRewardWrapper(env)
        # env = OurRewardWrapper(env)
        return env
    return _init

In [73]:
policy_kwargs = dict(activation_fn=th.nn.ReLU,
                     net_arch=dict(pi=[256, 512, 256, 128, 64], vi=[256, 512, 256, 128, 64]))


In [74]:
lr_schedule = get_linear_fn(start=0.001, end=0.00001, end_fraction=0.1)

num_envs = 25
envs = DummyVecEnv([make_env("ALE/Qbert-ram-v5") for _ in range(num_envs)])

model = PPO(
    "MlpPolicy",
    envs,
    verbose=1,
    n_steps = 1024,
    batch_size=64,
    ent_coef= 0.02,
    learning_rate=lr_schedule,
    policy_kwargs=policy_kwargs
    )

Using cuda device


/usr/local/lib/python3.11/dist-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


In [75]:
class WandbCallback(BaseCallback):
    def __init__(self, verbose=0):
        super(WandbCallback, self).__init__(verbose)
        self.episode_rewards = []
        self.episode_lengths = []

    def _on_step(self) -> bool:
        """Viene chiamato ad ogni step, logga i dati degli episodi."""

        for info in self.locals["infos"]:
            if "episode" in info:
                self.episode_rewards.append(info["episode"]["r"])
                self.episode_lengths.append(info["episode"]["l"])

        if len(self.episode_rewards) > 0:
            wandb.log({
                "rollout/ep_rew_mean": sum(self.episode_rewards) / len(self.episode_rewards),
                "rollout/ep_len_mean": sum(self.episode_lengths) / len(self.episode_lengths),
                "train/loss": self.model.logger.name_to_value.get("train/loss", 0),
                "train/policy_gradient_loss": self.model.logger.name_to_value.get("train/policy_gradient_loss", 0),
                "train/value_loss": self.model.logger.name_to_value.get("train/value_loss", 0),
            })

        return True


In [76]:
env = gym.make("ALE/Qbert-ram-v5")
env = ObsRewardWrapper(env)

eval_callback = EvalCallback(env, best_model_save_path="./BestModels/",
                             log_path="./logs/", eval_freq=500,
                             deterministic=True, render=False)
wandb_callback =  WandbCallback()

model.learn(total_timesteps = 5_000_000, callback=[eval_callback, wandb_callback])

wandb.finish()

# Valutazione del modello
mean_reward, std_reward = evaluate_policy(model, env, n_eval_episodes=10)
print(f"Ricompensa media: {mean_reward:.2f}, deviazione standard: {std_reward:.2f}")

/usr/local/lib/python3.11/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)
/usr/local/lib/python3.11/dist-packages/stable_baselines3/common/evaluation.py:67: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


Output streaming troncato alle ultime 5000 righe.
|    clip_fraction        | 0.0694      |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.62       |
|    explained_variance   | 0.832       |
|    learning_rate        | 1e-05       |
|    loss                 | 1.02        |
|    n_updates            | 590         |
|    policy_gradient_loss | -0.00783    |
|    value_loss           | 1.43        |
-----------------------------------------
Eval num_timesteps=1525000, episode_reward=2.20 +/- 0.98
Episode length: 568.80 +/- 22.70
---------------------------------
| eval/              |          |
|    mean_ep_length  | 569      |
|    mean_reward     | 2.2      |
| time/              |          |
|    total_timesteps | 1525000  |
---------------------------------
--------------------------------
| time/              |         |
|    fps             | 443     |
|    iterations      | 60      |
|    time_elapsed    | 3466    |
|    total_timesteps | 1536000 |
----

Ricompensa media: 15.70, deviazione standard: 3.96


In [81]:
env = gym.make("ALE/Qbert-ram-v5", render_mode="rgb_array")
env = ObsRewardWrapper(env)

In [82]:
# DummyVecEnv per compatibilità con Stable-Baselines3
env = DummyVecEnv([lambda: env])

In [83]:
# Registra video
video_folder = "./videos/"
env = VecVideoRecorder(
    env,               # Ambiente
    video_folder,      # Cartella per salvare i video
    record_video_trigger=lambda x: x % 10000000 == 0,  # Registra ogni 1000 passi
    video_length=10000000 # Durata massima del video in passi
)

In [84]:
# Resetta l'ambiente per registrare un episodio
obs = env.reset()

# Registra 3 episodi
for episode in range(3):
    obs = env.reset()
    for _ in range(10000000):  # Durata massima dell'episodio
        action, _states = model.predict(obs, deterministic=True)
        obs, rewards, dones, info = env.step(action)
        if dones[0]:  # L'episodio è terminato
            break

env.close()  # Salva il video

Moviepy - Building video /content/Q-Bert_RL/videos/rl-video-step-0-to-step-10000000.mp4.
Moviepy - Writing video /content/Q-Bert_RL/videos/rl-video-step-0-to-step-10000000.mp4



Moviepy - Done !
Moviepy - video ready /content/Q-Bert_RL/videos/rl-video-step-0-to-step-10000000.mp4


Moviepy - Building video /content/Q-Bert_RL/videos/rl-video-step-0-to-step-10000000.mp4.
Moviepy - Writing video /content/Q-Bert_RL/videos/rl-video-step-0-to-step-10000000.mp4



Moviepy - Done !
Moviepy - video ready /content/Q-Bert_RL/videos/rl-video-step-0-to-step-10000000.mp4
